We import all required modules

In [ ]:
from datetime import datetime
import os

import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import yaml
import markdown
from anyio import Path

from gcnn.training import set_up_callbacks, fit_model, setup_transform
from gcnn.utils import count_model_parameters
from gcnn.features import set_up_features
from gcnn.model import graph_potential
from gcnn.graph_database import CachedGraphDataset
from gcnn.paths import CONFIG_DIR, OUTPUT_DIR, PROCESSED_DIR, ensure_output_dir


We load the experiment configuration from sample.yml. All model hyperparameters and training settings are defined there.

In [ ]:
# Load experiment configuration
input_file = str(CONFIG_DIR / "sample.yml")

with open(input_file, "r") as input_stream:
    input_data = yaml.load(input_stream, Loader=yaml.Loader)

nGraphConvolutionLayers = input_data.get("nGraphConvolutionLayers", 1)
nFullyConnectedLayers = input_data.get("nFullyConnectedLayers", 1)
nMaxNeighbours = input_data.get("nMaxNeighbours", 6)
useCovalentRadii = input_data.get("useCovalentRadii", False)
nodeFeatures = input_data.get("nodeFeatures", ["atomic_number"])
nTotalNodeFeatures = input_data.get("nTotalNodeFeatures", 10)
nNeurons = input_data.get("nNeurons", None)

# Read transform preference from YAML (e.g. "standardize", "scale", or False)
transformData = input_data.get("transformData", False)

We read and set up the edge, bond-angle and dihedral-angle features we work with

In [ ]:
edges, bond_angle, dihedral_angle = set_up_features(input_data)

edge_parameters = edges.parameters()
bond_angle_parameters = bond_angle.parameters()

if dihedral_angle:
    dihedral_angle_parameters = dihedral_angle.parameters()

# now the total number of edge features is given by the sum
# of edges features + 2 * bond_angle features + dihedral_angle features
# all these features define the edge entries

nTotalEdgeFeatures = edges.n_features() + 2 * bond_angle.n_features()
if dihedral_angle:
    nTotalEdgeFeatures += dihedral_angle.n_features()

Now, we define our model

In [ ]:
model = graph_potential(
    n_gc_layers=nGraphConvolutionLayers,
    n_fc_layers=nFullyConnectedLayers,
    n_node_features=nTotalNodeFeatures,
    n_edge_features=nTotalEdgeFeatures,
    n_neurons=nNeurons or 15,
)

Escribimos cosas con logtext (DUDA AQUÍ)

In [ ]:
log_text = "\n# Model Description  \n"
log_text += "- nGraphConvolutionLayers: " + repr(nGraphConvolutionLayers) + "  \n"
log_text += "- nFullyConnectedLayers: " + repr(nFullyConnectedLayers) + "  \n"
if useCovalentRadii:
    log_text += "- Using Covalent Radii to find neighbours \n"
else:
    log_text += "- nMaxNeighbours: " + repr(nMaxNeighbours) + "  \n"
log_text += "- nTotalNodeFeatures: " + repr(nTotalNodeFeatures) + "  \n"
log_text += "- physical nodeFeatures: " + repr(nodeFeatures) + "  \n"
log_text += "- nEdgeFeatures: " + repr(nTotalEdgeFeatures) + "  \n"
log_text += "- Edge r_min: " + repr(edge_parameters["x_min"]) + " Angstrom  \n"
log_text += "- Edge r_max: " + repr(edge_parameters["x_max"]) + " Angstrom  \n"
log_text += "- Edge nFeatures: " + repr(edge_parameters["n_features"]) + " \n"
log_text += "- Edge sigma: " + repr(edge_parameters["sigma"]) + " Angstrom  \n"
log_text += "- Bond-Angle min: " + repr(bond_angle_parameters["x_min"]) + " radians  \n"
log_text += "- Bond-Angle max: " + repr(bond_angle_parameters["x_max"]) + " radians  \n"
log_text += "- Bond-Angle nFeatures: " + repr(bond_angle_parameters["n_features"]) + " \n"
log_text += "- Bond-Angle sigma: " + repr(bond_angle_parameters["sigma"]) + " radians  \n"
log_text += "- Bond-Angle normalised: " + repr(bond_angle_parameters["norm"]) + "   \n"
if dihedral_angle:
    log_text += "- Dihedral-Angle min: " + repr(dihedral_angle_parameters["x_min"]) + " radians  \n"
    log_text += "- Dihedral-Angle max: " + repr(dihedral_angle_parameters["x_max"]) + " radians  \n"
    log_text += "- Dihedral-Angle nFeatures: " + repr(dihedral_angle_parameters["n_features"]) + " \n"
    log_text += "- Dihedral-Angle sigma: " + repr(dihedral_angle_parameters["sigma"]) + " radians  \n"
    log_text += "- Dihedral-Angle normalised: " + repr(dihedral_angle_parameters["norm"]) + "   \n"

nParameters = count_model_parameters(model)
log_text += "- This model contains a total of: " + repr(nParameters) + " adjustable parameters  \n"

nEpochs = input_data.get("nEpochs", 100)
nBatch = input_data.get("nBatch", 50)
chkptFreq = input_data.get("nCheckpoint", 10)
learningRate = input_data.get("learningRate", 1.0e-3)
seed = input_data.get("randomSeed", 42)
nTrainMaxEntries = input_data.get("nTrainMaxEntries", None)
nValMaxEntries = input_data.get("nValMaxEntries", None)

log_text += "# Relaxation process  \n"
log_text += "- nEpochs: " + repr(nEpochs) + "  \n"
log_text += "- nBatch: " + repr(nBatch) + "  \n"
log_text += "- Checkpointing model and optimizer every " + repr(chkptFreq) + " epochs  \n"
log_text += "- learningRate: " + repr(learningRate) + "  \n"
log_text += "- random seed: " + repr(seed) + "  \n"

graphType = input_data.get("graphType", "covalent")
log_text += "- graph construction style: " + graphType + "  \n"

if nTrainMaxEntries:
    log_text += "- Size of training database: " + repr(nTrainMaxEntries) + "  \n"
else:
    log_text += "- Using full training database \n"
if nValMaxEntries:
    log_text += "- Size of validation/test database: " + repr(nValMaxEntries) + "  \n"
else:
    log_text += "- Using full validation/test database  \n"

Se pueden incluir fuerzas o no, de hecho no está implementado aunque igual estaría bien ponerlo cómo posible evolución del programa para poder predecir dinámica molecular por ejemplo.

Lo de loadModel sí está implementado y sirve para cargar un modelo si nos hemos quedado a mitad de simulación o similar

In [ ]:
calculateForces = input_data.get("calculateForces", False)
loadModel = input_data.get("loadModel", False)

log_text += "- Using forces in fitting: " + repr(calculateForces) + "  \n"


if loadModel:
    loadModelFileName = str(loadModel)
    if not Path(loadModelFileName).is_file():
        raise FileNotFoundError(f"Cannot load model: file '{loadModelFileName}' not found")
    log_text += "- Starting from previous model: " + loadModelFileName + "  \n"
else:
    log_text += "- Initialising model parameters from scratch   \n"

In [ ]:
descriptionText = input_data.get("descriptionText", " ")

descriptionText += log_text

Now, we prepare the graph dataset

In [ ]:
# Load pre-computed graphs from disk (built by notebook 1).
# All graphs live in RAM during training — no per-epoch I/O or recomputation.
trainDataset = CachedGraphDataset(
    PROCESSED_DIR / "train_graphs.pt", config=input_data
)
valDataset = CachedGraphDataset(
    PROCESSED_DIR / "val_graphs.pt", config=input_data
)
testDataset = CachedGraphDataset(
    PROCESSED_DIR / "test_graphs.pt", config=input_data
)

# Build transform from training data statistics (if requested in YAML)
transform = setup_transform(transformData, trainDataset)
if transform is not None:
    trainDataset.transform = transform
    valDataset.transform = transform
    testDataset.transform = transform

nTrain = len(trainDataset)
nValidation = len(valDataset)

trainLoader = DataLoader(trainDataset, batch_size=nBatch, num_workers=0)
valLoader = DataLoader(valDataset, batch_size=nBatch, num_workers=0)


We define a tensorboard writer to monitor the fitting process

In [ ]:
timeString = datetime.now().strftime("%d%m%Y-%H%M%S")
fileName = repr(nMaxNeighbours) + "nn-" + repr(nGraphConvolutionLayers) + "gcl-" + repr(nFullyConnectedLayers) + "fcl"
ensure_output_dir()
logFile = str(OUTPUT_DIR) + "/" + fileName + "-" + timeString

saveModelFileName = (
    "GraphPotential-"
    + repr(nMaxNeighbours)
    + "nn-"
    + repr(nGraphConvolutionLayers)
    + "gcl-"
    + repr(nFullyConnectedLayers)
    + "fcl-"
    + timeString
    + ".tar"
)

writer = SummaryWriter(logFile)

description = markdown.markdown(descriptionText)

writer.add_text("Description", description)

Optimizer

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learningRate)

# if we are to use a pre-saved model and optimizer, load their parameters here

if loadModel:

    # model = torch.load( loadModelFileName )

    checkpoint = torch.load(loadModelFileName)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    n_start = checkpoint["epoch"]

else:

    n_start = 0

lossFunction = torch.nn.MSELoss()

Now, we set-up the callbacks, if any

In [ ]:
anyCallBacks = input_data.get("callbacks", None)

callbacks = set_up_callbacks(anyCallBacks, optimizer)

Training process

In [ ]:
print("epoch      train-loss      validation-loss")
print("------------------------------------------")

fit_model(
    nEpochs,
    model,
    lossFunction,
    optimizer,
    nTrain,
    nValidation,
    trainLoader,
    valLoader,
    n_epoch_0=n_start,
    calculate_forces=calculateForces,
    weight=1.0e0,
    writer=writer,
    callbacks=callbacks,
    check_point_path=saveModelFileName,
    check_point_freq=chkptFreq,
    loss_path=logFile + "-loss.csv")

Let us build a histogram of the errors

In [ ]:
testLoader = DataLoader(testDataset, batch_size=1, num_workers=0)

print("         TEST SAMPLE ENERGIES             ")
print("------------------------------------------")

e = []
p = []

with torch.no_grad():

    file = open(logFile + "prediction.csv", "w")
    file.write("exact, predicted" + os.linesep)

    for sample in testLoader:
        
        energy = model(sample.x, sample.edge_index, sample.edge_attr, sample.batch)
        e.append(sample.y.item())
        p.append(energy.item())
        txt = repr(energy.item()) + ", " + repr(sample.y.item())

        file.write(txt + os.linesep)
        #print(txt)

    file.close()
prediction = np.array(p)
exact = np.array(e)
error = prediction - exact

if writer is not None:

    eMax = np.max(exact)
    eMin = np.min(exact)

    x = np.linspace(eMin, eMax, 100)

    fig, ax = plt.subplots()
    #figPredVsExN = plt.figure()

    ax.scatter(exact, prediction, color = '#0B00A8', label="Model predictions")
    ax.plot(x, x, color = 'red', label="Exact")

    ax.set_ylabel(r'Predicted energies', fontdict = {'fontsize':26, 'color':'k'})
    ax.set_xlabel(r'QM9 theoretical energies', fontdict = {'fontsize':24, 'color':'k'})
    
    ax.legend()

    plt.show()

    writer.add_figure("Prediction vs. exact ", fig, nEpochs)
    writer.add_histogram("Distribution of errors normalised data (prediction - exact)", error)

writer.close()